In [5]:
import os
import sys
import platform
import logging
from typing import Dict, Any

# Cấu hình Logging theo chuẩn PEP8
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger("EnvCheck")

def check_system_info() -> Dict[str, str]:
    """Thu thập thông tin hệ điều hành và phiên bản Python."""
    info = {
        "OS": platform.system(),
        "OS Version": platform.version(),
        "Architecture": platform.machine(),
        "Python Version": sys.version.split()[0],
        "Executable Path": sys.executable
    }
    return info

def check_gpu_support() -> Dict[str, Any]:
    """Kiểm tra khả năng tăng tốc phần cứng bằng PyTorch và TensorFlow."""
    gpu_info = {
        "pytorch_installed": False,
        "pytorch_cuda_available": False,
        "pytorch_device_count": 0,
        "pytorch_device_name": "N/A",
        "tensorflow_installed": False,
        "tensorflow_gpu_available": False
    }
    
    # Kiểm tra PyTorch
    try:
        import torch
        gpu_info["pytorch_installed"] = True
        gpu_info["pytorch_cuda_available"] = torch.cuda.is_available()
        if torch.cuda.is_available():
            gpu_info["pytorch_device_count"] = torch.cuda.device_count()
            gpu_info["pytorch_device_name"] = torch.cuda.get_device_name(0)
    except ImportError as e:
        logger.warning(f"PyTorch chưa được cài đặt đúng: {e}")

    # Kiểm tra TensorFlow
    try:
        import tensorflow as tf
        gpu_info["tensorflow_installed"] = True
        gpus = tf.config.list_physical_devices('GPU')
        gpu_info["tensorflow_gpu_available"] = len(gpus) > 0
    except ImportError as e:
        logger.warning(f"TensorFlow chưa được cài đặt đúng: {e}")

    return gpu_info

def check_core_libraries() -> Dict[str, str]:
    """Kiểm tra phiên bản các thư viện Quant & AI cốt lõi."""
    libraries = [
        "numpy", "pandas", "scipy", "sklearn", "xgboost", 
        "lightgbm", "catboost", "optuna", "yfinance", "ta", 
        "plotly", "streamlit", "fastapi"
    ]
    installed_versions = {}
    for lib in libraries:
        try:
            module = __import__(lib)
            installed_versions[lib] = getattr(module, "__version__", "Unknown")
        except ImportError:
            installed_versions[lib] = "NOT INSTALLED"
    return installed_versions

# --- THỰC THI KIỂM TRA ---
if __name__ == "__main__":
    logger.info("=== 1. HỆ THỐNG & PYTHON ===")
    sys_info = check_system_info()
    for k, v in sys_info.items():
        print(f"  - {k}: {v}")

    logger.info("=== 2. KIỂM TRA GPU / CUDA ===")
    gpu_status = check_gpu_support()
    print(f"  - PyTorch Installed: {gpu_status['pytorch_installed']}")
    print(f"  - PyTorch CUDA Available: {gpu_status['pytorch_cuda_available']}")
    print(f"  - PyTorch GPU Count: {gpu_status['pytorch_device_count']}")
    print(f"  - PyTorch GPU Name: {gpu_status['pytorch_device_name']}")
    print(f"  - TensorFlow Installed: {gpu_status['tensorflow_installed']}")
    print(f"  - TensorFlow GPU Available: {gpu_status['tensorflow_gpu_available']}")

    logger.info("=== 3. THƯ VIỆN CỐT LÕI ===")
    lib_status = check_core_libraries()
    for lib, ver in lib_status.items():
        status_icon = "✓" if ver != "NOT INSTALLED" else "✗"
        print(f"  [{status_icon}] {lib:<15}: {ver}")

2026-07-29 22:08:05,111 [INFO] === 1. HỆ THỐNG & PYTHON ===
  - OS: Windows
  - OS Version: 10.0.26200
  - Architecture: AMD64
  - Python Version: 3.10.20
  - Executable Path: c:\Users\chung\miniconda3\envs\stock_ai\python.exe
2026-07-29 22:08:05,114 [INFO] === 2. KIỂM TRA GPU / CUDA ===
2026-07-29 22:08:05,121 [WARNING] PyTorch chưa được cài đặt đúng: No module named 'torch'
2026-07-29 22:08:05,125 [WARNING] TensorFlow chưa được cài đặt đúng: No module named 'tensorflow'
  - PyTorch Installed: False
  - PyTorch CUDA Available: False
  - PyTorch GPU Count: 0
  - PyTorch GPU Name: N/A
  - TensorFlow Installed: False
  - TensorFlow GPU Available: False
2026-07-29 22:08:05,126 [INFO] === 3. THƯ VIỆN CỐT LÕI ===
  [✓] numpy          : 1.26.4
  [✓] pandas         : 2.2.0
  [✓] scipy          : 1.15.3
  [✓] sklearn        : 1.7.2
  [✓] xgboost        : 3.2.0
  [✗] lightgbm       : NOT INSTALLED
  [✗] catboost       : NOT INSTALLED
  [✗] optuna         : NOT INSTALLED
  [✗] yfinance       : N